# Monty Hall with $N$ doors

*others · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/monty_hall)

A car hides behind one of $N$ doors. You pick one; the host, who knows where the car is, opens $N-2$ doors with goats and offers you the last closed door. Millions of games for every $N$ from 3 to 1000 run at once in one $(B, G)$ tensor: switching wins with probability $(N-1)/N$ (the famous $2/3$ for three doors), staying with $1/N$, and every running win rate stays inside its binomial error band.

**Vector form**

$$P(\text{stay}) = P(p = c) = \frac1N,\qquad P(\text{switch}) = P(k = c) = P(p \ne c) = \frac{N-1}{N}$$

**Index (tensor) form**

$$k_{bg} = \begin{cases} c_{bg} & p_{bg} \ne c_{bg} \\ \text{random door} \ne p_{bg} & p_{bg} = c_{bg} \end{cases},\qquad \hat p_{bn} = \frac1n\sum_{g \le n} [k_{bg} = c_{bg}],\qquad \lvert \hat p_{bn} - p_b\rvert \lesssim 2\sqrt{\frac{p_b(1-p_b)}{n}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $N_b$ | `door_counts(max_doors)` | (B,) | the numbers of doors compared side by side, from 3 to 1000 |
| $c_{bg},\ p_{bg}$ | `(torch.rand(B, G) * N[:, None]).long()` | (B, G) | car door and first pick of game $g$ in row $b$, uniform over the $N_b$ doors; broadcasting gives every row its own $N_b$ |
| $k_{bg}$ | `torch.where(pick != car, car, other)` | (B, G) | the door the host leaves shut: forced to be the car when your pick is a goat, a random door when your pick is the car |
| $O_{bgd}$ | `opened_doors(pick, kept, N)` | (B, G, N) | the $N-2$ doors the host opens: every door except $p$ and $k$ |
| $[p = c],\ [k = c]$ | `outcomes(car, pick, kept, host)` | (B, G) | stay wins and switch wins; exactly one of them is true in every game |
| $\hat p_{bn}$ | `running_rate(wins, valid)` | (B, G) | cumulative wins over cumulative valid games: the win rate after every game |
| $2\sqrt{p(1-p)/n}$ | `binomial_band(p, n)` | (G,) | the $\pm 2\sigma$ band of a binomial proportion, which the running rate stays inside about 95 % of the time |

Switching loses only if your first pick was already the car, which happens with probability $1/N$; every other time the host's choice is forced, so the door he leaves shut carries the information of all the doors he opened. With 1000 doors the answer is obvious, and it is the same argument as with three. The puzzle became famous through Marilyn vos Savant's 1990 magazine column, when many readers, mathematicians among them, insisted that the answer was $1/2$. That intuition is right for a different game: choose the ignorant host, who opens doors at random and sometimes reveals the car. Throwing those games away (only a fraction $2/N$ survives) leaves stay and switch at exactly $1/2$, because a random opening says nothing about your door. The terminal also checks that no first pick is special, and a slow game-by-game reference in the tests agrees with the tensor version.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/monty_hall/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import exact_rates


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    host, g = params["host"], params["n_games"]
    doors = tensors.door_counts(int(params["max_doors"]))  # (B,)
    res = solve.play(doors, g, host, 0, dev)  # running rates (B, G)
    stay_th, switch_th = exact_rates(doors, host)
    print(f"{host} host: {len(doors)} door counts × {g:,} games = {len(doors) * g:,} games in one (B, G) tensor")
    for b, n in enumerate(doors.tolist()):
        extra = f"   valid games {res['valid'][b]:.4f} (theory {2 / n:.4f})" if host == "ignorant" else ""
        print(
            f"N = {n:4d}: stay {res['stay'][b, -1]:.4f} (theory {stay_th[b]:.4f})   "
            f"switch {res['switch'][b, -1]:.4f} (theory {switch_th[b]:.4f}){extra}"
        )
    by_pick = solve.rate_by_first_pick(res["picks"][0], res["switch_wins"][0], res["valid_games"][0], 3)
    print(f"N = 3, switch win rate by first pick: {[round(x, 4) for x in by_pick.tolist()]} (no door is special)")
    if host == "knowing":
        print("switching loses only when the first pick was already the car: probability 1/N")
    else:
        print("an ignorant host carries no information: when the car survives, stay and switch are a coin flip")
    return plot.animate(doors, res, host, out_dir / "monty_hall.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/monty_hall/equation.py
r"""The game → tensor translation.

There are N doors, one car and N − 1 goats. The contestant picks door p; the host, who knows where the car c is,
opens N − 2 of the other doors, all goats, and leaves one other door k shut. Stay or switch?

    \text{stay wins} = [p = c],\qquad \text{switch wins} = [k = c].

Since the host must leave the car shut whenever p ≠ c, switching wins exactly when the first pick was wrong:

    P(\text{stay}) = \frac1N,\qquad P(\text{switch}) = \frac{N-1}{N}      (1/3 and 2/3 for N = 3).

For G games the measured rate \hat p_n after n games is a prefix mean of 0/1 outcomes, a binomial proportion with
standard error \sqrt{p(1-p)/n}; the 2σ band holds the curve about 95 % of the time.

An ignorant host opens N − 2 random doors; the games in which he reveals the car are void (valid = 0). Given that
the car is still behind a shut door, stay and switch are equally likely to win: 1/2 each.
"""

import torch


def outcomes(
    car: torch.Tensor, pick: torch.Tensor, kept: torch.Tensor, host: str
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Stay wins [p = c], switch wins [k = c] and valid games (car not revealed), all bool of shape (B, G)."""
    stay = pick == car
    switch = kept == car
    valid = stay | switch  # the car is behind one of the two shut doors (always true for a knowing host)
    return stay, switch, valid


def opened_doors(pick: torch.Tensor, kept: torch.Tensor, n_doors: int) -> torch.Tensor:
    """Door mask O_{bgd} = [d ≠ p_{bg}][d ≠ k_{bg}]: the N − 2 doors the host opens.   (B, G) → (B, G, N)"""
    d = torch.arange(n_doors, device=pick.device)  # (N,)
    return (d != pick[..., None]) & (d != kept[..., None])


def running_rate(wins: torch.Tensor, valid: torch.Tensor | None = None) -> torch.Tensor:
    r"""\hat p_{bn} = \sum_{g \le n} w_{bg} / \sum_{g \le n} v_{bg}: win rate after every game.   (B, G) → (B, G)"""
    valid = torch.ones_like(wins) if valid is None else valid
    hits = torch.cumsum((wins & valid).double(), dim=-1)
    played = torch.cumsum(valid.double(), dim=-1)
    return hits / played.clamp(min=1)


def binomial_band(p: torch.Tensor, n: torch.Tensor) -> torch.Tensor:
    r"""2σ = 2\sqrt{p(1-p)/n}: the half-width of the 95 % band around the exact rate p after n games."""
    return 2 * torch.sqrt(p * (1 - p) / n)


def valid_fraction(doors: torch.Tensor, host: str) -> torch.Tensor:
    """Share of games that count: 1 for a knowing host, 2/N for an ignorant one (car still shut).   (B,) → (B,)"""
    return torch.ones(len(doors), dtype=torch.float64) if host == "knowing" else 2 / doors.double()


def exact_rates(doors: torch.Tensor, host: str) -> tuple[torch.Tensor, torch.Tensor]:
    """Theory: (1/N, (N − 1)/N) for a knowing host, (1/2, 1/2) for an ignorant one.   (B,) → (B,), (B,)"""
    n = doors.double()
    if host == "knowing":
        return 1 / n, (n - 1) / n
    return torch.full_like(n, 0.5), torch.full_like(n, 0.5)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/monty_hall/tensors.py
"""Build the concrete tensors: millions of games, one row per number of doors, played all at once."""

import torch


def door_counts(max_doors: int) -> torch.Tensor:
    """N_b: the numbers of doors compared side by side, from the classic 3 up to max_doors: shape (B,)."""
    ladder = [3, 4, 5, 10, 20, 50, 100, 200, 500, 1000]
    return torch.tensor([n for n in ladder if n <= max_doors])


def games(
    doors: torch.Tensor, n_games: int, host: str, seed: int, device: torch.device
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Car door c_{bg}, first pick p_{bg} and the other door the host leaves shut k_{bg}, all of shape (B, G).

    Row b plays G games with N_b doors. `floor(u · N)` turns a uniform u ∈ [0, 1) into a uniform door 0 … N − 1,
    and broadcasting N_b against (B, G) gives every row its own number of doors without a loop.
    """
    gen = torch.Generator().manual_seed(seed)
    n = doors.to(torch.float64)[:, None]  # (B, 1)
    u = torch.rand(3, len(doors), n_games, generator=gen, dtype=torch.float64)  # (3, B, G)
    car = (u[0] * n).long()  # (B, G) where the car is
    pick = (u[1] * n).long()  # (B, G) the contestant's first choice
    other = (pick + 1 + (u[2] * (n - 1)).long()) % doors[:, None]  # (B, G) a uniform door other than the pick
    if host == "knowing":
        # The host opens N − 2 goat doors. If the pick is a goat, the only door he can leave shut is the car;
        # if the pick is the car, every other door is a goat and he leaves a random one shut.
        kept = torch.where(pick != car, car, other)
    else:
        kept = other  # an ignorant host opens N − 2 random doors (other than the pick) and may reveal the car
    return car.to(device), pick.to(device), kept.to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/monty_hall/solve.py
"""Play every game, then follow the stay and switch win rates as the games accumulate."""

import torch

from . import tensors
from .equation import outcomes, running_rate


def play(doors: torch.Tensor, n_games: int, host: str, seed: int, device: torch.device) -> dict[str, torch.Tensor]:
    """Running stay/switch rates (B, G) and the fraction of valid games (B,) for every number of doors."""
    car, pick, kept = tensors.games(doors, n_games, host, seed, device)
    stay, switch, valid = outcomes(car, pick, kept, host)
    return {
        "stay": running_rate(stay, valid),
        "switch": running_rate(switch, valid),
        "valid": valid.double().mean(dim=-1),
        "picks": pick,
        "switch_wins": switch,
        "valid_games": valid,
    }


def final_rates(doors: torch.Tensor, n_games: int, host: str, seed: int, device: torch.device) -> torch.Tensor:
    """Win rates after all games: row 0 stay, row 1 switch, shape (2, B)."""
    res = play(doors, n_games, host, seed, device)
    return torch.stack([res["stay"][:, -1], res["switch"][:, -1]])


def rate_by_first_pick(picks: torch.Tensor, wins: torch.Tensor, valid: torch.Tensor, n_doors: int) -> torch.Tensor:
    """Switch win rate split by the first pick, over the valid games of one row: (G,) ×3 → (n_doors,). Flat."""
    onehot = torch.nn.functional.one_hot(picks, n_doors).double() * valid.double()[:, None]  # (G, N)
    return torch.einsum("gn,g->n", onehot, wins.double()) / onehot.sum(0)  # repeated g ⇒ summed


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/monty_hall/plot.py
"""Win rates converging inside their binomial bands, and the final rates against the number of doors."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .equation import binomial_band, exact_rates, valid_fraction


def animate(doors: torch.Tensor, res: dict, host: str, path: Path) -> Path:
    """doors (B,), res['stay'] and res['switch'] running rates (B, G)."""
    g = res["stay"].shape[1]
    curve = np.unique(np.geomspace(1, g, 400).astype(int))  # log-spaced game counts for the curves
    shown = np.unique(np.geomspace(10, g, 90).astype(int))  # game counts shown frame by frame
    stay, switch = res["stay"].cpu().numpy(), res["switch"].cpu().numpy()
    stay_th, switch_th = (t.numpy() for t in exact_rates(doors, host))
    ns = doors.numpy()
    colours = plt.cm.Blues(np.linspace(0.5, 1.0, len(ns)))
    share = valid_fraction(doors, host).numpy()  # the band narrows with the number of games that count
    n_curve = torch.tensor(curve, dtype=torch.float64)

    fig, (ax_c, ax_n) = plt.subplots(1, 2, figsize=(9.6, 4.3), gridspec_kw={"width_ratios": [1.3, 1]})
    picked = sorted({0, len(ns) // 2, len(ns) - 1})  # a few door counts on the left, all of them on the right
    lines = []
    for b in picked:
        band = binomial_band(torch.tensor(switch_th[b]), n_curve * share[b]).numpy()
        ax_c.fill_between(curve, switch_th[b] - band, switch_th[b] + band, color=colours[b], alpha=0.15, lw=0)
        ax_c.axhline(switch_th[b], color=colours[b], ls="--", lw=0.8)
        (line,) = ax_c.semilogx([], [], color=colours[b], lw=1.6, label=f"switch, N = {ns[b]}")
        lines.append((line, switch[b]))
    band = binomial_band(torch.tensor(stay_th[0]), n_curve * share[0]).numpy()
    ax_c.fill_between(curve, stay_th[0] - band, stay_th[0] + band, color="#f59e0b", alpha=0.18, lw=0)
    ax_c.axhline(stay_th[0], color="#f59e0b", ls="--", lw=0.8)
    (line,) = ax_c.semilogx([], [], color="#ea580c", lw=1.6, label=f"stay, N = {ns[0]}")
    lines.append((line, stay[0]))
    ax_c.set_xlim(1, g)
    ax_c.set_ylim(0, 1.02)
    ax_c.set_xlabel("games played n")
    ax_c.set_ylabel("win rate")
    ax_c.set_title("running win rate (dashed: exact, band: ±2σ)", fontsize=10)
    ax_c.legend(fontsize=8, frameon=False, loc="center right")

    nn = np.geomspace(3, max(ns.max(), 4), 200)
    if host == "knowing":
        ax_n.semilogx(nn, (nn - 1) / nn, color="#2563eb", lw=1.2, ls="--", label="switch: (N − 1)/N")
        ax_n.semilogx(nn, 1 / nn, color="#ea580c", lw=1.2, ls="--", label="stay: 1/N")
    else:
        ax_n.semilogx(nn, 0.5 + 0 * nn, color="#6b7280", lw=1.2, ls="--", label="1/2")
    ax_n.set_ylim(-0.02, 1.02)
    ax_n.set_xlim(2.5, nn[-1] * 1.4)
    ax_n.set_xlabel("number of doors N")
    ax_n.legend(fontsize=8, frameon=False, loc="center right")
    title = ax_n.set_title("", fontsize=10)
    markers: list = []

    def update(k: int):
        n = shown[k]
        mask = curve <= n
        for line, data in lines:
            line.set_data(curve[mask], data[curve[mask] - 1])
        for m in markers:
            m.remove()
        markers.clear()
        for rates, th, fmt, col in ((switch, switch_th, "o", "#2563eb"), (stay, stay_th, "s", "#ea580c")):
            err = binomial_band(torch.tensor(th), torch.tensor(n * share).clamp(min=1)).numpy()
            markers.append(ax_n.errorbar(ns, rates[:, n - 1], yerr=err, fmt=fmt, color=col, ms=5, capsize=3))
        title.set_text(f"after {n:,} games per N ({host} host)")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(shown), 90), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_games': 500000,
    'max_doors': 1000,
    'host': 'knowing'
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.monty_hall")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.monty_hall.main")
problem_dir = Path("problems/others/monty_hall")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
